In [1]:
#Create the main Repo list
import os
import pandas as pd
from urllib.parse import urlparse

# === INPUT / OUTPUT ===
SRC_CSV = r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ1\Clone_Status.csv"
OUT_DIR = r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\1_Main_Spreadsheet"
OUT_CSV = os.path.join(OUT_DIR, "3.2_Total_Repo.csv")
os.makedirs(OUT_DIR, exist_ok=True)

# === Load ===
df = pd.read_csv(SRC_CSV, dtype=str).fillna("")
df.columns = [c.strip() for c in df.columns]

# --- Find columns ---
def pick_col(candidates, cols):
    cols_lower = {c.lower(): c for c in cols}
    for cand in candidates:
        if cand.lower() in cols_lower:
            return cols_lower[cand.lower()]
    return None

clone_col = pick_col(["clone_status"], df.columns)
yml_col   = pick_col(["yml_detected"], df.columns)
url_col   = pick_col(["html_url"], df.columns)

if not clone_col or not yml_col or not url_col:
    missing = [name for name, col in {"clone_status": clone_col, "yml_detected": yml_col, "html_url/htm_url": url_col}.items() if not col]
    raise ValueError(f"Missing required column(s): {', '.join(missing)}")

# --- Normalize "yes" detection ---
def is_yes(x: str) -> bool:
    return str(x).strip().lower() in {"yes", "true", "y", "1"}

filtered = df[ df[clone_col].apply(is_yes) & df[yml_col].apply(is_yes) ].copy()

# --- Build full_name = owner.repo ---
def url_to_full_name(u: str) -> str:
    try:
        path = urlparse(str(u).strip()).path.strip("/")
        if not path:
            return ""
        if path.endswith(".git"):
            path = path[:-4]
        parts = path.split("/")
        if len(parts) >= 2:
            return f"{parts[0]}.{parts[1]}"
        return path
    except Exception:
        return ""

# Ensure full_name is lowercase
filtered["full_name"] = filtered[url_col].apply(url_to_full_name).str.lower()

# --- Keep only URL + full_name ---
out = filtered[[url_col, "full_name"]].rename(columns={url_col: "html_url"})

# Drop duplicates
out = out.drop_duplicates(subset=["html_url"]).reset_index(drop=True)

# Save
out.to_csv(OUT_CSV, index=False)
print(f"Saved: {OUT_CSV} (rows={len(out)})")


Saved: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\1_Main_Spreadsheet\3.2_Total_Repo.csv (rows=4518)


In [2]:
#Adds the Instru_tests for each repo

import os
import pandas as pd
from collections import defaultdict

# === PATHS ===
REPO_CSV = r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\1_Main_Spreadsheet\3.2_Total_Repo.csv"
TEST_DIR = r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ1\All_Test_Files"
OUT_CSV  = r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\1_Main_Spreadsheet\3.2_Total_Repo.csv"

# === Load repo list ===
repos = pd.read_csv(REPO_CSV, dtype=str).fillna("")
if "full_name" not in repos.columns:
    raise ValueError("Expected column 'full_name' in 3.1_Total_Repo.csv (format: owner.repo).")
repos["full_name"] = repos["full_name"].astype(str).str.strip()

# === Detection keywords ===
INSTRU_HINTS_NATIVE  = [
    "instrumentation", "androidtest", "connectedandroidtest",
    "espresso", "uiautomator", "orchestrator", "manageddevices", "gmd"
]
INSTRU_HINTS_FLUTTER = [
    "flutter", "dart"
]

def is_instru_file(fname: str) -> bool:
    name = fname.lower()
    return any(k in name for k in INSTRU_HINTS_NATIVE + INSTRU_HINTS_FLUTTER)

def classify_test(fname: str) -> str:
    lname = fname.lower()
    if any(k in lname for k in INSTRU_HINTS_FLUTTER):
        return "flutter"
    if any(k in lname for k in INSTRU_HINTS_NATIVE):
        return "native"
    return ""

# === Count tests per repo ===
native_counts  = defaultdict(int)
flutter_counts = defaultdict(int)

for fname in os.listdir(TEST_DIR):
    fpath = os.path.join(TEST_DIR, fname)
    if not os.path.isfile(fpath):
        continue
    if "__" not in fname:
        continue

    repo_token = fname.split("__", 1)[0].strip()
    if not repo_token:
        continue
    if not is_instru_file(fname):
        continue

    kind = classify_test(fname)
    if kind == "flutter":
        flutter_counts[repo_token] += 1
    elif kind == "native":
        native_counts[repo_token] += 1

# === Merge into repo DataFrame ===
repos["native_instru_test"]  = repos["full_name"].map(lambda k: native_counts.get(k, 0)).astype(int)
repos["flutter_instru_test"] = repos["full_name"].map(lambda k: flutter_counts.get(k, 0)).astype(int)
repos["Instru_test"] = (repos["native_instru_test"] + repos["flutter_instru_test"] > 0)

# === Save ===
repos.to_csv(OUT_CSV, index=False)
print(f"Saved: {OUT_CSV} (rows={len(repos)})")


Saved: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\1_Main_Spreadsheet\3.2_Total_Repo.csv (rows=4518)


the following cell is adjusted to includes two more new column created in yaml v4.0 for detecting flutter signal and device

In [3]:
# Aggregate instru_t_ci_signal and per-platform YML counts, append to main (case-insensitive)
import os
import re
import pandas as pd

BASE_DIR = r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\1_Main_Spreadsheet"
MAIN_CSV = os.path.join(BASE_DIR, "3.2_Total_Repo.csv")
YML_CSV  = os.path.join(BASE_DIR, "3.1.1_YML_FilesV6.0.csv")
OUT_CSV  = MAIN_CSV  # overwrite

def normalize_name(s: pd.Series) -> pd.Series:
    return s.astype(str).str.strip().str.lower()

def sanitize_col(name: str) -> str:
    s = re.sub(r"\W+", "_", str(name).strip().lower())
    s = re.sub(r"_+", "_", s).strip("_")
    return s or "unknown"

def to_bool_series(s: pd.Series) -> pd.Series:
    truthy = {"true","1","yes","y","t"}
    falsy  = {"false","0","no","n","f"}
    s = s.astype(str).str.strip().str.lower()
    return s.map(lambda x: True if x in truthy else (False if x in falsy else False)).astype("boolean")

def dedup_preserve(seq):
    seen, out = set(), []
    for x in seq:
        x = str(x)
        if x and x not in seen:
            seen.add(x); out.append(x)
    return out

# --- Load main ---
if not os.path.isfile(MAIN_CSV):
    raise FileNotFoundError(f"Main CSV not found: {MAIN_CSV}")
main = pd.read_csv(MAIN_CSV, dtype=str).fillna("")
if "full_name" not in main.columns:
    raise ValueError("Main CSV must contain 'full_name'.")
main["full_name_lc"] = normalize_name(main["full_name"])

# --- Load YML ---
if not os.path.isfile(YML_CSV):
    raise FileNotFoundError(f"YML CSV not found: {YML_CSV}")
yml = pd.read_csv(YML_CSV, dtype=str).fillna("")
for col in ("full_name", "ci_platform"):
    if col not in yml.columns:
        raise ValueError(f"{os.path.basename(YML_CSV)} must contain '{col}'.")

# ensure expected columns exist
for col, default in (
    ("instru_t_ci_signal", ""),
    ("confidence", ""),
    ("confidence_reason", ""),
    # NEW: Flutter columns expected from detector; create safe defaults if absent
    ("flutter_integ_t_signal", "false"),
    ("flutter_integ_t_d", ""),
    # NEW: Environment + Invocation aggregation sources
    ("execution_environment", ""),
    ("test_invocation", ""),
):
    if col not in yml.columns:
        yml[col] = default

yml["full_name_lc"] = normalize_name(yml["full_name"])

# --- Aggregate per-platform YML counts and totals ---
plat_counts = (
    yml.groupby(["full_name_lc", "ci_platform"])
       .size()
       .unstack(fill_value=0)
)
if not plat_counts.empty:
    plat_counts = plat_counts.rename(columns={c: sanitize_col(c) for c in plat_counts.columns})
    # collapse potential duplicates caused by sanitation
    plat_counts = plat_counts.T.groupby(level=0).sum().T
    plat_counts = plat_counts.reset_index()
else:
    plat_counts = pd.DataFrame(columns=["full_name_lc"])

total_counts = (
    yml.groupby("full_name_lc").size().rename("Total_YMLs").reset_index()
)

# --- Aggregate instru_t_ci_signal (True if any row True) ---
yml["instru_t_ci_signal"] = to_bool_series(yml["instru_t_ci_signal"])
agg_flag = (
    yml[["full_name_lc", "instru_t_ci_signal"]]
      .groupby("full_name_lc", as_index=False)["instru_t_ci_signal"]
      .any()
)
agg_flag["instru_t_ci_signal"] = agg_flag["instru_t_ci_signal"].astype("boolean")

# --- Aggregate confidence + confidence_reason per repo ---
yml["conf_norm"] = yml["confidence"].str.strip().str.lower()
def aggregate_confidence(group: pd.DataFrame) -> pd.Series:
    confs = set(group["conf_norm"].dropna().tolist())
    if "high" in confs:
        level = "high"
    elif "medium" in confs:
        level = "medium"
    elif "low" in confs:
        level = "low"
    else:
        level = ""

    if level:
        reasons = group.loc[group["conf_norm"] == level, "confidence_reason"].astype(str).str.strip()
        reasons = [r for r in reasons if r]
        agg_reason = " || ".join(dedup_preserve(reasons))
    else:
        agg_reason = ""

    return pd.Series({"instru_t_ci_confidence": level, "confidence_reason": agg_reason})

agg_conf = (
    yml.groupby("full_name_lc")
       .apply(aggregate_confidence)
       .reset_index()
)

# --- NEW: Aggregate Flutter columns ---
# repo-level flutter_integ_t_signal: True if any row True
yml["flutter_integ_t_signal"] = to_bool_series(yml["flutter_integ_t_signal"])
flutter_sig = (
    yml.groupby("full_name_lc", as_index=False)["flutter_integ_t_signal"]
       .any()
)
flutter_sig["flutter_integ_t_signal"] = flutter_sig["flutter_integ_t_signal"].astype("boolean")

# Device token normalization map and nulls
ALIASES_DEV = {
    # windows
    "win": "windows", "windows-latest": "windows", "windows-2019": "windows", "windows-2022": "windows",
    # linux
    "ubuntu": "linux", "ubuntu-20.04": "linux", "ubuntu-22.04": "linux", "debian": "linux", "linux-latest": "linux",
    # mac
    "osx": "macos", "darwin": "macos", "macos-latest": "macos", "macos-13": "macos", "macos-14": "macos",
    # ios
    "iphoneos": "ios", "apple-ios": "ios",
    # android (+ common typo)
    "android-os": "android", "andorid": "android",
    # web
    "chrome": "web", "browser": "web",
}
NULS = {"", "nan", "none", "null"}

# repo-level flutter_integ_t_d: tokenize, normalize, dedupe, SORT, and join
yml["flutter_integ_t_d_norm"] = yml["flutter_integ_t_d"].astype(str).str.strip().str.lower()

SEP_RE = re.compile(r"[,\s;/|/]+")
# NEW: For env/inv we want to keep multi-word phrases together; don't split on whitespace.
SEP_MULTI = re.compile(r"[,\|;//]+")

def tokenize_devices(cell: str):
    if not cell or cell in NULS:
        return []
    toks = [t.strip().lower() for t in SEP_RE.split(cell) if t.strip()]
    normed = []
    for t in toks:
        t = ALIASES_DEV.get(t, t)
        # normalize substrings that contain "android"
        if "andorid" in t: t = "android"
        if "android" in t: t = "android"
        normed.append(t)
    return normed

def agg_flutter_devices(group: pd.DataFrame) -> pd.Series:
    bag = set()
    for v in group["flutter_integ_t_d_norm"]:
        for tok in tokenize_devices(v):
            if tok not in NULS:
                bag.add(tok)
    # Sort alphabetically; join with comma+space
    out_list = sorted(bag)
    return pd.Series({"flutter_integ_t_d": ", ".join(out_list)})

flutter_dev = (
    yml.groupby("full_name_lc")
       .apply(agg_flutter_devices)
       .reset_index()
)

# --- NEW: Aggregate execution_environment and test_invocation ---
def tokenize_multi(cell: str):
    """Split on comma/semicolon/pipe/slash, keep multi-word phrases intact, strip."""
    if cell is None:
        return []
    s = str(cell).strip()
    if not s:
        return []
    s_l = s.lower()
    if s_l in NULS or s_l == "unknown":
        return []
    return [t.strip() for t in SEP_MULTI.split(s) if t.strip() and t.strip().lower() != "unknown"]

def agg_env_inv(group: pd.DataFrame) -> pd.Series:
    env_seen = {}
    inv_seen = {}

    for v in group["execution_environment"].astype(str):
        for tok in tokenize_multi(v):
            key = tok.lower()
            if key not in env_seen:
                env_seen[key] = tok  # preserve first-seen casing

    for v in group["test_invocation"].astype(str):
        for tok in tokenize_multi(v):
            key = tok.lower()
            if key not in inv_seen:
                inv_seen[key] = tok  # preserve first-seen casing

    env_out = ", ".join([env_seen[k] for k in sorted(env_seen.keys())])
    inv_out = ", ".join([inv_seen[k] for k in sorted(inv_seen.keys())])

    return pd.Series({
        "execution_environment": env_out,
        "test_invocation": inv_out
    })

agg_envinv = (
    yml.groupby("full_name_lc")
       .apply(agg_env_inv)
       .reset_index()
)

# --- Merge aggregates together ---
agg = (total_counts
       .merge(plat_counts, on="full_name_lc", how="left")
       .merge(agg_flag,    on="full_name_lc", how="left")
       .merge(agg_conf,    on="full_name_lc", how="left")
       .merge(flutter_sig, on="full_name_lc", how="left")    # NEW
       .merge(flutter_dev, on="full_name_lc", how="left")    # NEW
       .merge(agg_envinv,  on="full_name_lc", how="left"))   # NEW

# --- Merge into main (case-insensitive) ---
out = main.merge(agg, on="full_name_lc", how="left").drop(columns=["full_name_lc"])

# Normalize numeric/platform columns to int (fill NaN with 0)
platform_cols = [c for c in plat_counts.columns if c != "full_name_lc"]
for c in platform_cols:
    if c not in out.columns:
        out[c] = 0
    out[c] = pd.to_numeric(out[c], errors="coerce").fillna(0).astype(int)

if "Total_YMLs" not in out.columns:
    out["Total_YMLs"] = 0
out["Total_YMLs"] = pd.to_numeric(out["Total_YMLs"], errors="coerce").fillna(0).astype(int)

# Final instru_t_ci_signal in main: aggregated value
existing = out["instru_t_ci_signal"] if "instru_t_ci_signal" in out.columns else pd.Series([False]*len(out), index=out.index)
existing = to_bool_series(existing).fillna(False)
yml_flag = out.get("instru_t_ci_signal", pd.Series([False]*len(out), index=out.index))
yml_flag = to_bool_series(yml_flag).fillna(False)
out["instru_t_ci_signal"] = yml_flag.astype("boolean")

# --- Ensure Flutter columns exist with correct types/defaults ---
if "flutter_integ_t_signal" not in out.columns:
    out["flutter_integ_t_signal"] = False
out["flutter_integ_t_signal"] = to_bool_series(out["flutter_integ_t_signal"]).fillna(False).astype("boolean")

if "flutter_integ_t_d" not in out.columns:
    out["flutter_integ_t_d"] = ""
out["flutter_integ_t_d"] = out["flutter_integ_t_d"].astype(str).fillna("").str.strip()

# --- Ensure NEW columns exist and are blank if missing ---
for c in ("execution_environment", "test_invocation"):
    if c not in out.columns:
        out[c] = ""
    out[c] = out[c].astype(str).fillna("").str.strip()

# Save
out.to_csv(OUT_CSV, index=False)
print(f"Saved: {OUT_CSV} (rows={len(out)})")
print(f"instru_t_ci_signal True={int(out['instru_t_ci_signal'].sum())} / {len(out)}")
print("Platform columns added:", [c for c in platform_cols if c in out.columns])
print(f"flutter_integ_t_signal True={int(out['flutter_integ_t_signal'].sum())} / {len(out)}")
# NEW: quick peek counts of env/inv populated
print("execution_environment non-blank:", int((out['execution_environment'].astype(str).str.strip()!='').sum()))
print("test_invocation non-blank:", int((out['test_invocation'].astype(str).str.strip()!='').sum()))


C:\Users\gilla\AppData\Local\Temp\ipykernel_14056\3082089332.py:117: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(aggregate_confidence)
C:\Users\gilla\AppData\Local\Temp\ipykernel_14056\3082089332.py:179: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(agg_flutter_devices)


Saved: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\1_Main_Spreadsheet\3.2_Total_Repo.csv (rows=4518)
instru_t_ci_signal True=455 / 4518
Platform columns added: ['appveyor', 'azure_pipelines', 'bitbucket', 'bitrise', 'circle_ci', 'cirrus', 'codemagic', 'github_actions', 'gitlab', 'semaphore', 'travis_ci']
flutter_integ_t_signal True=0 / 4518
execution_environment non-blank: 396
test_invocation non-blank: 327


C:\Users\gilla\AppData\Local\Temp\ipykernel_14056\3082089332.py:222: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(agg_env_inv)


In [4]:
# adds the instru testing signals config and unit test signals both CI and Configs

import os
import pandas as pd

BASE_DIR = r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\1_Main_Spreadsheet"

FILES = {
    "instru_t_signal_config": ("3.1.1_Instru_T_Signal_ConfigV6.0.csv",
                               ["instru_t_signal_config", "instru_test_signal_config"]),
    "unit_t_signal_ci":       ("3.1.2_Unit_T_Signal_CI.csv",
                               ["unit_t_signal_ci", "unit_test_signal_ci"]),
    "unit_t_signal_config":   ("3.1.2_Unit_T_Signal_Config.csv",
                               ["unit_t_signal_config", "unit_test_config_signal"]),
}

MAIN_CSV = os.path.join(BASE_DIR, "3.2_Total_Repo.csv")
OUT_CSV  = MAIN_CSV  # overwrite

def normalize_name(s: pd.Series) -> pd.Series:
    return s.astype(str).str.strip().str.lower()

def to_bool_series(s: pd.Series) -> pd.Series:
    truthy = {"true","1","yes","y","t"}
    falsy  = {"false","0","no","n","f"}
    s = s.astype(str).str.strip().str.lower()
    return s.map(lambda x: True if x in truthy else (False if x in falsy else False)).astype("boolean")

def load_and_aggregate_bool(path: str, out_col: str, candidates: list[str]) -> pd.DataFrame:
    if not os.path.isfile(path):
        print(f"[WARN] Missing file -> {os.path.basename(path)}; skipping {out_col}.")
        return pd.DataFrame(columns=["full_name", out_col])
    df = pd.read_csv(path, dtype=str).fillna("")
    if "full_name" not in df.columns:
        print(f"[WARN] 'full_name' not in {os.path.basename(path)}; skipping {out_col}.")
        return pd.DataFrame(columns=["full_name", out_col])

    df["full_name"] = normalize_name(df["full_name"])
    flag_col = next((c for c in candidates if c in df.columns), None)
    if flag_col is None:
        print(f"[WARN] None of {candidates} found in {os.path.basename(path)}; skipping {out_col}.")
        return pd.DataFrame(columns=["full_name", out_col])

    b = to_bool_series(df[flag_col])
    agg = (
        pd.DataFrame({"full_name": df["full_name"], out_col: b})
          .groupby("full_name", as_index=False)[out_col]
          .any()
    )
    agg[out_col] = agg[out_col].astype("boolean")
    return agg

# --- Load & normalize main ---
if not os.path.isfile(MAIN_CSV):
    raise FileNotFoundError(f"Main CSV not found: {MAIN_CSV}")

main = pd.read_csv(MAIN_CSV, dtype=str).fillna("")
if "full_name" not in main.columns:
    raise ValueError("Main CSV must contain a 'full_name' column.")
main["full_name"] = normalize_name(main["full_name"])

# --- Merge aggregated flags (default False; True if any repo hit) ---
for out_col, (fname, aliases) in FILES.items():
    agg_path = os.path.join(BASE_DIR, fname)
    agg = load_and_aggregate_bool(agg_path, out_col, aliases)

    # ensure column exists in main with default False (nullable boolean)
    if out_col not in main.columns:
        main[out_col] = pd.Series([False] * len(main), index=main.index, dtype="boolean")
    else:
        # coerce any existing values safely -> boolean, unknowns -> False
        main[out_col] = to_bool_series(main[out_col])

    if not agg.empty:
        lookup = agg.set_index("full_name")[out_col]
        upd = main["full_name"].map(lookup).astype("boolean").fillna(False)
        # OR (True wins)
        main[out_col] = (main[out_col] | upd).astype("boolean")

# --- Save ---
main.to_csv(OUT_CSV, index=False)
print(f"Saved: {OUT_CSV} (rows={len(main)})")

# Optional: quick counts
for col in FILES.keys():
    if col in main.columns:
        print(f"{col}: True={int(main[col].sum())} / {len(main)}")


Saved: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\1_Main_Spreadsheet\3.2_Total_Repo.csv (rows=4518)
instru_t_signal_config: True=2292 / 4518
unit_t_signal_ci: True=2162 / 4518
unit_t_signal_config: True=2302 / 4518


Check to add "GMD" if its test invocation exists

In [5]:
import os
import pandas as pd

# --- Paths (same folder) ---
BASE = r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\1_Main_Spreadsheet"
MAIN_CSV = os.path.join(BASE, "3.2_Total_Repo.csv")                         # MAIN file to update
GMD_CFG  = os.path.join(BASE, "3.1.1_Instru_T_Signal_ConfigV6.0.csv")       # repo list with gmd_present

# --- Safety checks ---
for p in (MAIN_CSV, GMD_CFG):
    if not os.path.isfile(p):
        raise FileNotFoundError(f"File not found: {p}")

# --- Load ---
df_main = pd.read_csv(MAIN_CSV, dtype=str)
df_cfg  = pd.read_csv(GMD_CFG,  dtype=str)

# --- Required columns ---
req_main = {"full_name", "execution_environment"}
req_cfg  = {"full_name", "gmd_present"}

missing_main = req_main - set(df_main.columns)
missing_cfg  = req_cfg  - set(df_cfg.columns)
if missing_main:
    raise KeyError(f"Missing column(s) in main CSV: {sorted(missing_main)}")
if missing_cfg:
    raise KeyError(f"Missing column(s) in config CSV: {sorted(missing_cfg)}")

# --- Normalize join keys ---
df_main["_key"] = df_main["full_name"].astype(str).str.lower().str.strip()
df_cfg["_key"]  = df_cfg["full_name"].astype(str).str.lower().str.strip()

# --- Parse booleans ---
def parse_bool(x) -> bool:
    s = str(x).strip().lower() if x is not None else ""
    return s in {"true", "t", "1", "yes", "y"}

# Repo-level gmd_present (from config)
gmd_present_by_repo = (
    df_cfg[["_key", "gmd_present"]]
    .assign(gmd_present_bool=lambda d: d["gmd_present"].apply(parse_bool))
    .groupby("_key")["gmd_present_bool"]
    .any()
)

# Mask rows in MAIN that belong to repos with gmd_present == True
mask_update = df_main["_key"].map(gmd_present_by_repo).fillna(False)

# Helper: append "GMD" to comma-separated env list (preserve order, avoid duplicates)
def add_gmd(env: str) -> str:
    current = "" if pd.isna(env) else str(env)
    if not current.strip():
        return "Emulator_GMD"
    parts = [p.strip() for p in current.split(",") if p.strip()]
    if not any(p.lower() == "emulator_gmd" for p in parts):
        parts.append("Emulator_GMD")
    return ",".join(parts)

rows_updated = int(mask_update.sum())
if rows_updated:
    df_main.loc[mask_update, "execution_environment"] = (
        df_main.loc[mask_update, "execution_environment"].apply(add_gmd)
    )

# Save (no backup)
df_main.drop(columns=["_key"], errors="ignore").to_csv(MAIN_CSV, index=False, encoding="utf-8-sig")

print(f"Repos with gmd_present=True: {gmd_present_by_repo.sum()}")
print(f"Rows updated (added Emulator_GMD): {rows_updated}")
print(f"CSV updated: {MAIN_CSV}")


Repos with gmd_present=True: 24
Rows updated (added Emulator_GMD): 24
CSV updated: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\1_Main_Spreadsheet\3.2_Total_Repo.csv


C:\Users\gilla\AppData\Local\Temp\ipykernel_14056\3588742748.py:47: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  mask_update = df_main["_key"].map(gmd_present_by_repo).fillna(False)


In [6]:
# -*- coding: utf-8 -*-
"""
Compute ci_android_signal:

- Non-Flutter: True iff (execution_environment signal present) OR (test_invocation signal present)
  (i.e., either column is non-unknown / non-blank)

- Flutter: keep handling unchanged (Android devices OR Android-ish runtime in reason;
          flip off when devices explicitly non-Android and no runtime cues)

Overwrites the input spreadsheet in place.
"""

import re
import pandas as pd
from pathlib import Path

# -------- CONFIG --------
MAIN_DIR = r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\1_Main_Spreadsheet"
PREFERRED_FILENAME = "3.2_Total_Repo.csv"

# -------- HELPERS --------
def find_main_file(folder: str) -> Path:
    folder = Path(folder)
    for pattern in [PREFERRED_FILENAME, "*.csv", "*.xlsx"]:
        matches = sorted(folder.glob(pattern))
        if matches:
            return matches[0]
    raise FileNotFoundError(f"No CSV/XLSX found in: {folder}")

def read_table(path: Path) -> pd.DataFrame:
    if path.suffix.lower() == ".csv":
        try:
            return pd.read_csv(path, low_memory=False)
        except UnicodeDecodeError:
            return pd.read_csv(path, encoding="latin-1", low_memory=False)
    return pd.read_excel(path, sheet_name=0)

def write_table(df: pd.DataFrame, path: Path) -> None:
    if path.suffix.lower() == ".csv":
        df.to_csv(path, index=False)
    else:
        df.to_excel(path, index=False)

def to_bool_series(s: pd.Series) -> pd.Series:
    return s.astype(str).str.strip().str.lower().isin({"1","true","t","yes","y","on"})

def get_col(df: pd.DataFrame, names):
    for n in ([names] if isinstance(names, str) else names):
        if n in df.columns:
            return n
    return None

# --- Runtime keywords (used by Flutter handling) ---
RX_AT_BOOST = re.compile(r'boosted\s*\(\s*androidtest\s+present\s*\)', re.I)
RX_RUNTIME = re.compile(
    r'(?:'
    r'gcloud\s+firebase|flank|saucectl|appcenter\s+test|maestro|browserstack|bstack|'
    r'connected(?:android)?test|connectedcheck|devicecheck|'
    r'(?:^|[\s:/.-])(?:gradlew?|gradle)\b|'
    r'\bam\s+instrument\b|\bandroidtest\b|'
    r'managed(?:virtual)?device|\bgmd\b|'
    r'reactivecircus/android-emulator-runner|reactivecircus\s+runner|\breactivecircus\b|'
    r'sdkmanager[^"\n]*system-images|\bavdmanager\b|'
    r'\bemulator\b[^\n]*?(?:-avd|@)\b|'
    r'\bapi[-_ ]?level\b|\babi/arch\b|\btarget\s+image\b'
    r')',
    re.I
)
def has_android_runtime(reason: str) -> bool:
    r = RX_AT_BOOST.sub("", str(reason or ""))
    return bool(RX_RUNTIME.search(r))

# --- Flutter device helpers (unchanged) ---
NULS = {"", "nan", "none", "null"}
SEP_RE = re.compile(r"[,\s;/|/]+")

def devices_nonblank_and_nonandroid(cell) -> bool:
    if pd.isna(cell):
        return False
    s = str(cell).strip().lower()
    if s in NULS:
        return False
    toks = [t for t in SEP_RE.split(s) if t]
    if not toks:
        return False
    return not any(("android" in t) or ("andorid" in t) for t in toks)

def has_android_in_devices(cell) -> bool:
    if pd.isna(cell):
        return False
    s = str(cell).strip().lower()
    return ("android" in s) or ("andorid" in s)

TOKEN_SPLIT = re.compile(r"[,\|;/]+")

def any_token_in(cell: str, allowed: set[str]) -> bool:
    if pd.isna(cell):
        return False
    toks = [t.strip().lower() for t in TOKEN_SPLIT.split(str(cell)) if t.strip()]
    return any(t in allowed for t in toks)

def label_flag_from_text(s: pd.Series) -> pd.Series:
    """
    Convert execution_environment / test_invocation label columns to 0/1.
    Treat anything NOT in {'', 'unknown', 'none', 'na', 'nan'} as positive.
    """
    t = s.astype(str).str.strip().str.lower()
    return (~t.isin({"", "unknown", "none", "na", "nan"})).astype(int)

# -------- CORE: compute ci_android_signal (as agreed) --------
def compute_ci_only(df: pd.DataFrame) -> pd.DataFrame:
    # Column names
    col_inv    = get_col(df, ["test_invocation_MR", "test_invocation"])
    col_env    = get_col(df, ["execution_environment_MR", "execution_environment"])
    col_reason = get_col(df, ["confidence_reason", "ci_confidence_reason", "ci_reason"])
    col_fsig   = get_col(df, ["flutter_integ_t_signal", "flutter_cli_signal", "flutter_signal"])
    col_fdev   = get_col(df, ["flutter_integ_t_d", "flutter_devices", "flutter_device_list"])

    # Base series (safe defaults)
    inv_col   = df[col_inv].astype(str)  .str.strip().str.lower() if col_inv else pd.Series("", index=df.index)
    env_col   = df[col_env].astype(str)  .str.strip().str.lower() if col_env else pd.Series("", index=df.index)
    reason    = df[col_reason].astype(str)                     if col_reason else pd.Series("", index=df.index)
    flutter_ci   = to_bool_series(df[col_fsig])                if col_fsig else pd.Series(False, index=df.index)
    flutter_dev  = df[col_fdev]                                if col_fdev else pd.Series("", index=df.index)

    # Non-Flutter rule: signal if either env or invocation is non-unknown
    # (accept common variants for env labels)
    env_flag = (~env_col.isin({"", "unknown", "none", "na", "nan"})).astype(int)
    inv_flag = (~inv_col.isin({"", "unknown", "none", "na", "nan"})).astype(int)
    base_signal = (env_flag.eq(1) | inv_flag.eq(1))

    # ---------- Flutter handling (unchanged) ----------
    reason_has_runtime      = reason.apply(has_android_runtime)
    flutter_on_android      = flutter_dev.apply(has_android_in_devices)
    flutter_nonA_nonblank   = flutter_dev.apply(devices_nonblank_and_nonandroid)

    # Flutter contributes to Android CI if Flutter CI ran AND devices include Android OR runtime shows Android
    flutter_androidish = flutter_ci & (flutter_on_android | reason_has_runtime)

    # Flip off when flutter devices explicitly non-Android AND no Android runtime cues
    no_android_runtime = ~reason_has_runtime
    flutter_flip = flutter_nonA_nonblank & no_android_runtime
    flutter_androidish = flutter_androidish & (~flutter_flip)

    # Final signal: non-Flutter uses base_signal; Flutter rows use guard
    ci_android_signal = base_signal.astype(bool)
    ci_android_signal[flutter_ci] = flutter_androidish[flutter_ci]

    out = df.copy()
    out["ci_android_signal"] = ci_android_signal.astype(bool)
    return out

def main():
    in_path = find_main_file(MAIN_DIR)
    print(f"[INFO] Using main spreadsheet: {in_path}")

    df = read_table(in_path)
    out_df = compute_ci_only(df)

    write_table(out_df, in_path)
    print(f"[OK] Overwrote: {in_path}")

    true_ct = int(pd.Series(out_df["ci_android_signal"]).sum())
    print(f"ci_android_signal True={true_ct} / {len(out_df)}")

if __name__ == "__main__":
    main()


[INFO] Using main spreadsheet: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\1_Main_Spreadsheet\3.2_Total_Repo.csv
[OK] Overwrote: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\1_Main_Spreadsheet\3.2_Total_Repo.csv
ci_android_signal True=462 / 4518
